# CA-Transformer Quick Start

Small transformers that infer a **cellular-automaton rule in context**. Each model sees the first $M$ frames of a trajectory generated by a rule it has *never seen during training*, then predicts the rest.

There are 12 released models ([Hugging Face collection](https://huggingface.co/collections/sdeangroup/spatial-induction-heads)):
3 tasks (1D $V{=}2$, 1D $V{=}3$, 2D von Neumann) × deterministic / stochastic rules × *Shared* / *Dedicated* architecture.

This notebook downloads **one** checkpoint (a few MB to ~50 MB) and runs in a few minutes on a CPU.
It covers six steps: install, load a model, generate an example, run inference, visualize, and try your own rule and context.

## 1. Install

In [ ]:
# Run from the root of a clone of the repository (where this notebook lives).
# Installs the dependencies; the code itself is the repo's src/ package.
%pip install -q -e .

## 2. Pick and load a model

`list_pretrained()` shows every released model. To try a different one, change `MODEL` and re-run the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src import (list_pretrained, load_pretrained, generate_example, generate_examples,
                 predict, rollout, evaluate, continue_trajectory, check_context,
                            plot_example, plot_prediction, plot_rollout, plot_ar_accuracy)

list_pretrained()

In [ ]:
MODEL = "1d-det-v2k3-shared"     # <- change me

bundle = load_pretrained(MODEL, device="auto")   # downloads only this checkpoint (cached afterwards)
bundle

`bundle.spec` holds everything about the task and architecture: the grid shape, the number of frames $T$, the context length $M$, the separator token ids, and so on.

In [ ]:
bundle.spec

## 3. Generate an example

By default the rule is drawn from the **held-out test pool**, meaning rules the model never trained on. For stochastic models it is a fresh random transition table. As in the test set, the initial state is resampled until every neighborhood configuration appears in the context, so the rule is fully identifiable.

In [ ]:
example = generate_example(bundle.spec, seed=0)
print("rule:", example.rule if not bundle.spec.stochastic else f"{example.rule.shape} transition table",
      "| split:", example.rule_split)
print("context:", example.context.shape, " full trajectory:", example.trajectory.shape)
plot_example(example);

You can also pick the rule yourself:
* **deterministic:** an integer (the Wolfram number for 1D $V{=}2$, e.g. `rule=110`) or a full output table of shape `(V**K,)`;
* **stochastic:** an array of shape `(V**K, V)` whose row $i$ is the next-state distribution for neighborhood configuration $i$.

In [ ]:
if bundle.spec.key.startswith("1d-det-v2k3"):
    example = generate_example(bundle.spec, rule=110, seed=0)   # rule 110 is held out: never seen in training
    print("rule 110 split:", example.rule_split)

## 4. Run inference

`predict` runs both evaluation modes:
* **teacher-forced:** a single forward pass over the ground-truth sequence; each cell is predicted from the true history;
* **autoregressive:** the model sees only the first $M$ frames and generates the rest itself (greedy for deterministic models, sampled for stochastic ones).

Metrics are computed on frames $M..T-1$. Deterministic models report cell and sequence accuracy. Stochastic models report $\mathrm{KL}(\text{reference} \,\|\, \text{model})$ against the true rule and against the causal add-one $k$-gram estimate from the context. These are the same definitions as the paper's evaluator.

In [ ]:
result = predict(bundle, example, mode="both")
result.metrics

## 5. Visualize

In [ ]:
plot_prediction(example, result);

How to read the plot:
* **1D:** time runs downward and the dashed line marks the end of the context.
* **2D:** frames with blue borders are context.
* **Deterministic models:** red boxes are wrong cells.
* **Stochastic models:** colors are probability-weighted mixes of the state colors, so for $V{=}2$ the shade is $P(\text{state}=1)$.

To get a more reliable number, score a batch of fresh held-out examples. For the full evaluation (100k fresh test trajectories, teacher-forced + autoregressive), run `python -m src.evaluate --model <key>`.

In [ ]:
batch = generate_examples(bundle.spec, n=64, seed=1)
batch_result = predict(bundle, batch)
{k: v for k, v in batch_result.metrics.items() if k != "ar/cell_acc_by_step"}

In [ ]:
if not bundle.spec.stochastic:
    plot_ar_accuracy(batch_result);

## 6. Try your own rule and context

Give the model your own input:
* `RULE` is the rule that generates the trajectory. For deterministic models it is a rule number (the Wolfram number for 1D $V{=}2$) or a `(V**K,)` output table. For stochastic models it is a `(V**K, V)` array of next-state probabilities.
* `CONTEXT` is the frames the model sees, shape `(m, *grid)` with states in `0..V-1`. Each frame must follow from the previous one under `RULE`. The models were trained with $m = M$.

The model only sees `CONTEXT`. `RULE` gives the ground truth: `continue_trajectory` simulates it forward from the last context frame, and the model's rollout is scored against that.

The defaults below are for the 1D $V{=}2$ deterministic models: rule 110 (held out from training) with a 4-frame context of 16 cells. For other models the cell falls back to a held-out rule and a simulated context.

In [ ]:
spec = bundle.spec

# ---- your inputs ------------------------------------------------------------
RULE = 110
CONTEXT = np.array([
    [1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0],
    [1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1],
    [1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0],
    [1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1],
])
# -----------------------------------------------------------------------------

if not spec.key.startswith("1d-det-v2k3"):      # defaults above don't fit this model
    ex = generate_example(spec, seed=7)          # held-out rule (random table if stochastic)
    RULE, CONTEXT = ex.rule, ex.trajectory[: spec.M]

check_context(spec, CONTEXT, RULE)

`check_context` reports whether every transition in `CONTEXT` is possible under `RULE` (`follows_rule`) and how many of the $V^K$ neighborhood configurations appear in it (`configs_seen`). Rule entries whose configuration never appears cannot be inferred from the context; the test set requires all of them.

In [ ]:
reference = continue_trajectory(spec, CONTEXT, RULE, num_frames=spec.T - len(CONTEXT), seed=0)
result = rollout(bundle, CONTEXT, reference=reference)
print(result.metrics)
plot_rollout(result);

Without a rule there is no ground truth, but the model still continues the context: `plot_rollout(rollout(bundle, CONTEXT))`. For stochastic rules, `continue_trajectory` gives one random realization per `seed`, and stochastic rollouts are sampled.

## Next steps
* Set `MODEL` to a 2D or stochastic model, e.g. `"2d-det-v2k5-dedicated"` or `"1d-stoch-v3k3-shared"`, and re-run the notebook.
* `evaluate(bundle, n=512)` gives a quick estimate of the paper metrics.
* The arrays behind every plot are on the result objects: `result.tf_probs`, `result.ar_states`, `result.ar_probs`.